# Compliance Data — Search without GPU

Public-only corpus: 38 frameworks → 1519 chunks (512/50). Works in Colab, no `torch`, no `pixi`.

**Data:** `data/silver/compliance_chunks.parquet` (1.6M) from [Releases](https://github.com/dataengineergaurav/open-regulatory-corpus/releases) or local `data/silver/`

This notebook shows: load → filter by framework → keyword search (TF-IDF, no embeddings) → cite source.

> Not legal advice. Check `source` URLs for authoritative text.

In [1]:
import pandas as pd
from pathlib import Path

# local or download from Release
parquet = Path("../data/silver/compliance_chunks.parquet")
if not parquet.exists():
    # fallback: download latest Release (no auth)
    url = "https://github.com/dataengineergaurav/open-regulatory-corpus/releases/latest/download/compliance-silver-2026-09-01.parquet"
    # try generic latest — if 404, use local path hint
    print(f"{parquet} not found — download from Releases: {url}")
    # uncomment to fetch:
    # import urllib.request; urllib.request.urlretrieve(url, "compliance_chunks.parquet"); parquet = Path("compliance_chunks.parquet")
else:
    df = pd.read_parquet(parquet)
    print(f"rows={len(df)} cols={list(df.columns)}")
    df.head(2)

rows=1519 cols=['framework_id', 'chunk_id', 'text', 'source', 'kind', 'token_est', 'sha256']


In [2]:
# distribution — what's actually in Silver
df["framework_id"].value_counts().head(15)

framework_id
CJIS-6.1_pdf      462
IRS-1075_pdf1     222
SOX_pdf1          155
CCPA-CPRA_pdf2    121
FDA-AI-MD          80
SOX_pdf2           67
EAR                52
FERPA              39
AU-PRIVACY-AI      38
OMB-M25-21         29
NIST-CSF2          28
EAR_pdf2           20
SG-MODEL-AI        19
OMB-M25-22         15
ONC-HTI1_pdf2      14
Name: count, dtype: int64

In [3]:
# filter by framework — e.g. FDA AI-enabled medical devices
fda = df[df.framework_id == "FDA-AI-MD"]
print(f"FDA-AI-MD: {len(fda)} chunks")
print(fda.iloc[0]["text"][:800])
print("source:", fda.iloc[0]["source"])

FDA-AI-MD: 80 chunks
Artificial Intelligence-Enabled Medical Devices The FDA encourages the development of innovative, safe, and effective medical devices, including devices that incorporate artificial intelligence (AI). The AI-Enabled Medical Device List is a resource intended to identify AI-enabled medical devices that are authorized for marketing in the United States. Digital health innovators can refer to this list to gain insights into the current device landscape and regulatory expectations, which can help foster innovation and ensure public safety. This list can also provide transparency for healthcare providers and patients to clearly identify when medical devices use AI technologies. Contents of the AI-Enabled Medical Devices List: - The devices in this list have met the FDA’s applicable premarket req
source: data/bronze/latest/raw/FDA-AI-MD.html


In [4]:
# keyword search without embeddings — TF-IDF (sklearn) or simple contains
# ponytail: sklearn if available, else pure pandas
query = "incident response logging"
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    X = vec.fit_transform(df["text"].tolist())
    q = vec.transform([query])
    scores = cosine_similarity(q, X).flatten()
    top = scores.argsort()[::-1][:5]
    for i in top:
        print(f"{scores[i]:.3f} {df.iloc[i]['framework_id']:15} {df.iloc[i]['chunk_id']}")
        print(df.iloc[i]["text"][:400].replace("\n"," ")[:400])
        print("—", df.iloc[i]["source"])
        print()
except ImportError:
    # fallback: substring
    hits = df[df.text.str.contains(query, case=False, na=False)].head(5)
    for _, r in hits.iterrows():
        print(r["framework_id"], r["chunk_id"])
        print(r["text"][:400])
        print("—", r["source"], "\n")

0.503 IRS-1075_pdf1   IRS-1075_pdf1-137
mechanisms can provide a push and/or pull capability for users to obtain incident response assistance. For example, individuals might have access to a website to query the assistance capability, or the assistance capability can proactively send information to users (general distribution or targeted) as part of increasing understanding of current response capabilities and support. Control Enhanceme
— data/bronze/latest/raw/IRS-1075_pdf1.pdf

0.500 IRS-1075_pdf1   IRS-1075_pdf1-134
Addresses purpose, scope, roles, responsibilities, management commitment, coordination among organizational entities and compliance; and (b) Is consistent with applicable laws, Executive Orders, directives, regulations, policies, standards, and guidelines; and 2. Procedures to facilitate the implementation of the incident response policy and the associated Incident response controls; b. Designate 
— data/bronze/latest/raw/IRS-1075_pdf1.pdf

0.495 CJIS-6.1_pdf    CJIS-6.1

In [5]:
# cross-framework example: NGO wants all privacy + AI governance chunks
privacy = ["CCPA-CPRA","FERPA","COPPA","AU-PRIVACY-AI","UK-DP-AI","DOJ-DSP"]
ai_gov = ["NIST-AI-RMF","NIST-AI-600-1","OMB-M25-21","OWASP-LLM-2026"]
subset = df[df.framework_id.isin(privacy + ai_gov)]
print(f"privacy+ai subset: {len(subset)} chunks")
subset["framework_id"].value_counts()

privacy+ai subset: 126 chunks


framework_id
FERPA             39
AU-PRIVACY-AI     38
OMB-M25-21        29
UK-DP-AI           7
NIST-AI-600-1      4
DOJ-DSP            3
CCPA-CPRA          2
COPPA              2
NIST-AI-RMF        1
OWASP-LLM-2026     1
Name: count, dtype: int64

## Next
- Replace TF-IDF with embeddings later: `sentence-transformers/all-MiniLM-L6-v2` + FAISS (Gold phase).
- Cite `source` + `sha256` per chunk — every answer should link to `framework_id` provenance in `data/sources.json`.
- WAF gaps: `HIPAA`, `CMMC`, `HHS-PART2`, `IL-AIVIA`, `EU-AI-ACT`, `GDPR`, `BR-LGPD` currently 0 chunks (see `docs/DATA_PIPELINE.md`).